# Greatest Hits — Train on Colab T4

Runs the v1 MLP from cached cochleagrams + visual features. **Does not need the 60 GB raw dataset** — only the ~2 GB cache bundle produced by `python src/prepare_colab_bundle.py` locally.

Before running:
1. Make sure `cache_bundle.tar.gz` is uploaded to Drive at `MyDrive/cv_final_project/cache_bundle.tar.gz` (or edit `BUNDLE_PATH` below).
2. Runtime → Change runtime type → **GPU** (T4 is fine).

In [ ]:
!nvidia-smi -L

In [ ]:
# Clone code from GitHub
!rm -rf /content/cv_final_project
!git clone https://github.com/Owen-M8/cv_final_project.git /content/cv_final_project
%cd /content/cv_final_project

In [ ]:
# Install Python deps. Colab already has torch/torchvision; pycochleagram needs the GitHub install.
!pip install -q --upgrade pip
!pip install -q soundfile librosa opencv-python tqdm
!pip install -q git+https://github.com/mcdermottLab/pycochleagram.git

In [ ]:
# Mount Google Drive and pull the cache bundle
from google.colab import drive
drive.mount('/content/drive')

BUNDLE_PATH = '/content/drive/MyDrive/cv_final_project/cache_bundle.tar.gz'
!ls -lh "$BUNDLE_PATH"

# Extract into project root. The archive contains a top-level cache/ directory.
!tar -xzf "$BUNDLE_PATH" -C /content/cv_final_project/
!ls /content/cv_final_project/cache/ | head -5
!echo "cochleagram files:  $(ls /content/cv_final_project/cache/ | grep _coch.npz | wc -l)"
!echo "feature files:      $(ls /content/cv_final_project/cache/ | grep _feat.npy | wc -l)"
!ls /content/cv_final_project/cache/clip_index.json

In [ ]:
# Sanity check: load the clip index, verify it matches the cache.
import sys
sys.path.insert(0, '/content/cv_final_project/src')
from dataset import load_clip_index
from pathlib import Path

train_clips, test_clips = load_clip_index()
print(f'{len(train_clips)} train clips, {len(test_clips)} test clips')

# Spot-check that the corresponding cache files exist
from visual_features import _feat_cache_path
from dataset import _cache_path
missing_feat = [c for c in train_clips[:200] if not _feat_cache_path(c).exists()]
missing_coch = [c.entry for c in train_clips[:200] if not _cache_path(c.entry.video_id).exists()]
print(f'spot-check (first 200 train clips): {len(missing_feat)} missing features, {len(missing_coch)} missing cochleagrams')

In [ ]:
# Train the v1 MLP. Should run in 5-10 minutes on T4.
%cd /content/cv_final_project
!python src/train.py --epochs 20 --batch-size 64

In [ ]:
# Inference on a held-out clip (first test clip).
# NOTE: inference.py needs to read the video file to extract visual features for new
# clips. To listen to predictions on Colab without uploading 60 GB, we use the cached
# features for an existing test clip and only need cochleagram inversion.
import sys
sys.path.insert(0, '/content/cv_final_project/src')
import numpy as np
import torch
import soundfile as sf
from pathlib import Path

from config import CHECKPOINT_DIR, OUTPUT_DIR, TARGET_SR, ENVELOPE_SR
from cochleagram import cochleagram_to_waveform
from inference import load_checkpoint
from visual_features import _feat_cache_path
from dataset import load_clip_index

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, pca = load_checkpoint(CHECKPOINT_DIR / 'v1.pt', device)

_, test_clips = load_clip_index()
OUT = Path('/content/cv_final_project/outputs/colab_predictions')
OUT.mkdir(parents=True, exist_ok=True)

for i, clip in enumerate(test_clips[:5]):
    feats = np.load(_feat_cache_path(clip))
    z = model(torch.from_numpy(feats).unsqueeze(0).to(device)).detach().cpu().numpy()[0]
    coch_flat = pca.inverse_transform(z[None])[0]
    coch = np.clip(coch_flat.reshape(pca.target_shape), 0.0, None).astype(np.float32)
    wav = cochleagram_to_waveform(coch, sr=TARGET_SR, envelope_sr=ENVELOPE_SR)
    out_path = OUT / f'{i:02d}_{clip.entry.video_id}_t{clip.onset_time:.2f}_{clip.material}.wav'
    sf.write(str(out_path), wav, TARGET_SR)
    print(out_path)

In [ ]:
# Listen to predictions inline
from IPython.display import Audio, display
from pathlib import Path
for p in sorted(Path('/content/cv_final_project/outputs/colab_predictions').glob('*.wav')):
    print(p.name)
    display(Audio(str(p)))

In [ ]:
# Save the trained checkpoint + predictions back to Drive
!mkdir -p /content/drive/MyDrive/cv_final_project/checkpoints
!cp -v checkpoints/v1.pt /content/drive/MyDrive/cv_final_project/checkpoints/
!cp -v checkpoints/v1_history.json /content/drive/MyDrive/cv_final_project/checkpoints/
!cp -rv outputs/colab_predictions /content/drive/MyDrive/cv_final_project/

---

# Three-stream V2A (real-world track)

ResNet50 + R(2+1)D-18 + RAFT flow + Transformer fusion + PCA head.
Uses a separate cache bundle from the V1 path.

**Before running:** upload `three_stream_bundle.tar` to
`MyDrive/cv_final_project/three_stream_bundle.tar` (~12 GB; created locally with
`find cache -maxdepth 1 \( -name '*_clip.npz' -o -name '*_flow.npy' \) -print0 | tar -cf cache/three_stream_bundle.tar --null -T -`).

The V1 `cache_bundle.tar.gz` from earlier in this notebook still needs to be
extracted first — the cochleagrams in it are the regression target.


In [ ]:
# Persist checkpoints to Drive directly so a disconnect doesn't lose work.
# Replaces the local checkpoints/ dir with a symlink into Drive.
import os
DRIVE_CKPT = '/content/drive/MyDrive/cv_final_project/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)
!rm -rf /content/cv_final_project/checkpoints
!ln -s "$DRIVE_CKPT" /content/cv_final_project/checkpoints
!ls -la /content/cv_final_project/checkpoints/

In [ ]:
# Extract the three-stream bundle (clip.npz + flow.npy files)
THREE_STREAM_BUNDLE = '/content/drive/MyDrive/cv_final_project/three_stream_bundle.tar'
!ls -lh "$THREE_STREAM_BUNDLE"
!tar -xf "$THREE_STREAM_BUNDLE" -C /content/cv_final_project/
!echo "clip.npz files:  $(ls /content/cv_final_project/cache/ | grep _clip.npz | wc -l)"
!echo "flow.npy files:  $(ls /content/cv_final_project/cache/ | grep _flow.npy  | wc -l)"


In [ ]:
# Train the full three-stream model on T4 (~5-10 min/epoch, ~2-3h total).
# `--num-workers 2` works on Colab (Linux fork is fine, unlike macOS).
%cd /content/cv_final_project
!python -u src/train_three_stream.py --epochs 20 --batch-size 32 --num-workers 2

In [ ]:
# Inference using the cached frames+flow (no raw video needed).
import sys, dataclasses
sys.path.insert(0, '/content/cv_final_project/src')
import numpy as np, torch, soundfile as sf
from pathlib import Path

from config import (
    CHECKPOINT_DIR, OUTPUT_DIR, TARGET_SR, ENVELOPE_SR, PCA_DIM,
    CLIP_FRAMES, MAX_TEMPORAL_JITTER,
)
from cochleagram import cochleagram_to_waveform
from streams import FrozenStreams, compute_frozen_features, load_streams
from three_stream_model import ModelConfig, PCAHead, ThreeStreamV2A
from model import PCAState
from dataset import load_clip_index

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt_path = CHECKPOINT_DIR / 'three_stream_app-m3d-flow.pt'
blob = torch.load(ckpt_path, map_location=device, weights_only=False)
cfg = ModelConfig(**blob['model_cfg'])
head = PCAHead(d_model=cfg.d_model, out_dim=PCA_DIM)
model = ThreeStreamV2A(head, streams=tuple(cfg.streams), cfg=cfg).to(device).eval()
model.load_state_dict(blob['model_state'])
p = blob['pca']
pca = PCAState(mean=p['mean'], components=p['components'],
               explained_variance=p['explained_variance'],
               target_shape=tuple(p['target_shape']))

frozen = FrozenStreams.build(device)
_, test_clips = load_clip_index()

OUT = Path('/content/cv_final_project/outputs/three_stream_predictions')
OUT.mkdir(parents=True, exist_ok=True)

for i, clip in enumerate(test_clips[:5]):
    frames_full, flow_full = load_streams(clip)
    s = MAX_TEMPORAL_JITTER  # canonical (un-jittered) window
    frames = frames_full[s:s + CLIP_FRAMES]
    flow = flow_full[s:s + CLIP_FRAMES - 1]

    frames_chw = (frames.astype(np.float32) / 255.0).transpose(0, 3, 1, 2)
    frames_t = torch.from_numpy(frames_chw).unsqueeze(0).to(device)
    flow_t = torch.from_numpy(flow.astype(np.float32)).unsqueeze(0).to(device)

    with torch.no_grad():
        app, m3d = compute_frozen_features(frames_t, frozen)
        z = model(appearance=app, motion3d=m3d, flow_field=flow_t).cpu().numpy()[0]

    coch_flat = pca.inverse_transform(z[None])[0]
    coch = np.clip(coch_flat.reshape(pca.target_shape), 0.0, None).astype(np.float32)
    wav = cochleagram_to_waveform(coch, sr=TARGET_SR, envelope_sr=ENVELOPE_SR)
    out_path = OUT / f'{i:02d}_{clip.entry.video_id}_t{clip.onset_time:.2f}_{clip.material}.wav'
    sf.write(str(out_path), wav, TARGET_SR)
    print(out_path)

In [ ]:
# Listen
from IPython.display import Audio, display
from pathlib import Path
for p in sorted(Path('/content/cv_final_project/outputs/three_stream_predictions').glob('*.wav')):
    print(p.name)
    display(Audio(str(p)))

In [ ]:
# Save predictions to Drive (checkpoints are already there via symlink)
!mkdir -p /content/drive/MyDrive/cv_final_project/three_stream_predictions
!cp -v outputs/three_stream_predictions/*.wav /content/drive/MyDrive/cv_final_project/three_stream_predictions/